In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix , auc, precision_recall_curve, roc_auc_score, balanced_accuracy_score
from scipy.stats import uniform, randint
from sklearn.model_selection import ParameterGrid, cross_validate
import matplotlib.pyplot as plt

In [13]:
dataset_gold = pd.read_csv('../../assets/gold/experimentos_finales/dataset_final.csv')

In [14]:
dataset_gold.isna().sum()

sexo                             0
año_inscripcion_facultad         0
edad_inscripcion                 2
target                           0
cohorte                          0
                              ... 
tiempo_desde_cbc                 0
carrera_encoded                  0
nivel_estudio_madre_encoded    445
nivel_estudio_padre_encoded    521
situacion_laboral_encoded       19
Length: 87, dtype: int64

In [15]:
max_arboles = [5, 10, 15, 20]
metricas = ["gini", "entropy"]
bootstrap = [True]
max_depth = [3]
min_samples_split= [2]
min_samples_leaf = [1]
max_features = ['sqrt']


def crear_grilla(max_arboles, metricas, bootstrap, min_samples_split, min_samples_leaf, max_features):
    parametros = {
        'n_estimators': max_arboles,
        'criterion': metricas,
        'bootstrap': bootstrap,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features
    }
    grilla = list(ParameterGrid(parametros))
    return grilla

grilla_parametros = crear_grilla(max_arboles, metricas, bootstrap,min_samples_split, min_samples_leaf, max_features)
grilla_parametros

[{'bootstrap': True,
  'criterion': 'gini',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 5},
 {'bootstrap': True,
  'criterion': 'gini',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 10},
 {'bootstrap': True,
  'criterion': 'gini',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 15},
 {'bootstrap': True,
  'criterion': 'gini',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 20},
 {'bootstrap': True,
  'criterion': 'entropy',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 5},
 {'bootstrap': True,
  'criterion': 'entropy',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 10},
 {'bootstrap': True,
  '

### Seleccion de variables

Otro modelo solo con las variables más importantes y obtener un mejor accuracy. Tomo el mejor experimento (según la métrica que tome) y lo vuelvo a correr solo con las importantes con Montecarlo cross validación.

Para el experimento 2 la configuración que mejor AUC dio fue:

{'bootstrap': True,
  'criterion': 'entropy',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 20}

Entonces puedo mirar las variables que fueron más importantes para ese caso y ver cómo da el AUC, usando igualmente el Monte Carlo Cross Validation.

Primero debo quedarme solamente con las columnas que contienen las variables de interes.

In [16]:
dataset_gold.columns

Index(['sexo', 'año_inscripcion_facultad', 'edad_inscripcion', 'target',
       'cohorte', 'nota_cluster_1.0', 'nota_cluster_2.0', 'nota_cluster_3.0',
       'nota_cluster_4.0', 'nota_cluster_5.0', 'nota_cluster_6.0',
       'cantidad_de_veces_que_rindio_cluster_1.0',
       'cantidad_de_veces_que_rindio_cluster_2.0',
       'cantidad_de_veces_que_rindio_cluster_3.0',
       'cantidad_de_veces_que_rindio_cluster_4.0',
       'cantidad_de_veces_que_rindio_cluster_5.0',
       'cantidad_de_veces_que_rindio_cluster_6.0',
       'fecha_promedio_cluster_1.0', 'fecha_promedio_cluster_2.0',
       'fecha_promedio_cluster_3.0', 'fecha_promedio_cluster_4.0',
       'fecha_promedio_cluster_5.0', 'fecha_promedio_cluster_6.0',
       'cursada_por_uba_xxi_cluster_1.0', 'cursada_por_uba_xxi_cluster_2.0',
       'cursada_por_uba_xxi_cluster_3.0', 'cursada_por_uba_xxi_cluster_4.0',
       'cursada_por_uba_xxi_cluster_5.0', 'cursada_por_uba_xxi_cluster_6.0',
       'finales_inscriptos_0.0', 'finales_in

In [17]:
columnas_importantes_20_arboles = ['tp_aprobados_3.0', 'tp_aprobados_2.0', 'finales_inscriptos_3.0', 'finales_inscriptos_2.0',
                                    'tp_aprobado_materia_6',
                                    'inscripciones_3.0', 'fecha_TP_materia_7', 'inscripciones_2.0', 'nota_final_materia_5',
                                    'tp_aprobado_materia_4', 'tp_aprobado_materia_3', 'nota_final_materia_7',
                                    'fecha_final_materia_5', 'nota_final_materia_3', 'tp_aprobado_materia_8',
                                    'fecha_final_materia_7', 'tp_aprobado_materia_5', 'finales_inscriptos_1.0', 'tp_aprobado_materia_7',
                                    'fecha_TP_materia_4', 'target']

dataset_gold_filtrado = dataset_gold[columnas_importantes_20_arboles]
dataset_gold_filtrado.isna().sum()

tp_aprobados_3.0             0
tp_aprobados_2.0             0
finales_inscriptos_3.0       0
finales_inscriptos_2.0       0
tp_aprobado_materia_6      879
inscripciones_3.0            0
fecha_TP_materia_7        1215
inscripciones_2.0            0
nota_final_materia_5      1596
tp_aprobado_materia_4      472
tp_aprobado_materia_3      331
nota_final_materia_7      1786
fecha_final_materia_5     1578
nota_final_materia_3      1440
tp_aprobado_materia_8     1532
fecha_final_materia_7     1754
tp_aprobado_materia_5      654
finales_inscriptos_1.0       0
tp_aprobado_materia_7     1215
fecha_TP_materia_4         472
target                       0
dtype: int64

In [18]:
resultados = []
importancias_resultados = []
params = {'bootstrap': True,
  'criterion': 'entropy',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 20}


accuracy_scores_train = []
accuracy_scores_test = []
balanced_accuracy_scores_train = []
balanced_accuracy_scores_test = []
auc_scores_train = []
auc_scores_test = []
importancias_atributos = []
print(f"Evaluando parámetros: {params}")
# Repetir el proceso 5 veces para obtener diferentes splits de entrenamiento y test
for i in range(5):
    # 1. Separar un 20% random como test, el random state cambia en cada iteracion, entonces da un 20% distinto en cada una
    X_train_base, X_test, y_train_base, y_test = train_test_split(dataset_gold_filtrado[dataset_gold_filtrado.columns.difference(['target'])],
                                                                    dataset_gold_filtrado['target'], test_size=0.2, random_state=i,
                                                                    stratify=dataset_gold_filtrado['target'])
    
    modelo = RandomForestClassifier(random_state=42, **params)
    modelo.fit(X_train_base, y_train_base)

    # Guardar importancia de atributos
    importancias = modelo.feature_importances_
    importancias_atributos.append(importancias)
    
    # Predicciones de clase
    y_train_pred = modelo.predict(X_train_base)
    y_validation_pred = modelo.predict(X_test)

    # Predicciones de probabilidad para AUC ROC (para la clase positiva)
    y_train_proba = modelo.predict_proba(X_train_base)[:, 1]
    y_validation_proba = modelo.predict_proba(X_test)[:, 1]

    # Accuracy
    acc_train = accuracy_score(y_train_base, y_train_pred)
    acc_test = accuracy_score(y_test, y_validation_pred)
    accuracy_scores_train.append(acc_train)
    accuracy_scores_test.append(acc_test)

    # Balanced Accuracy
    bal_acc_train = balanced_accuracy_score(y_train_base, y_train_pred)
    bal_acc_test = balanced_accuracy_score(y_test, y_validation_pred)
    balanced_accuracy_scores_train.append(bal_acc_train)
    balanced_accuracy_scores_test.append(bal_acc_test)

    # AUC ROC
    auc_train = roc_auc_score(y_train_base, y_train_proba)
    auc_test = roc_auc_score(y_test, y_validation_proba)
    auc_scores_train.append(auc_train)
    auc_scores_test.append(auc_test)

# Calcular promedios y desviaciones estándar
accuracy_train_mean = np.mean(accuracy_scores_train)
accuracy_test_mean = np.mean(accuracy_scores_test)
accuracy_train_std = np.std(accuracy_scores_train)
accuracy_test_std = np.std(accuracy_scores_test)
bal_accuracy_train_mean = np.mean(balanced_accuracy_scores_train)
bal_accuracy_test_mean = np.mean(balanced_accuracy_scores_test)
bal_accuracy_train_std = np.std(balanced_accuracy_scores_train)
bal_accuracy_test_std = np.std(balanced_accuracy_scores_test)
auc_train_mean = np.mean(auc_scores_train)
auc_test_mean = np.mean(auc_scores_test)
auc_train_std = np.std(auc_scores_train)
auc_test_std = np.std(auc_scores_test)
print(f"Resultados para {params}:")

resultados.append({
    'params': params,
    'bootstrap': params['bootstrap'],
    'criterion': params['criterion'],
    'n_estimators': params['n_estimators'],
    'train_accuracy_mean': accuracy_train_mean,
    'test_accuracy_mean': accuracy_test_mean,
    'train_accuracy_std': accuracy_train_std,
    'test_accuracy_std': accuracy_test_std,
    'train_balanced_accuracy_mean': bal_accuracy_train_mean,
    'test_balanced_accuracy_mean': bal_accuracy_test_mean,
    'train_balanced_accuracy_std': bal_accuracy_train_std,
    'test_balanced_accuracy_std': bal_accuracy_test_std,
    'train_auc_mean': auc_train_mean,
    'test_auc_mean': auc_test_mean,
    'train_auc_std': auc_train_std,
    'test_auc_std': auc_test_std
})

# Calcular importancia promedio y std
importancias_array = np.array(importancias_atributos)
importancia_media = np.mean(importancias_array, axis=0)
importancia_std = np.std(importancias_array, axis=0)

# Asignar nombres de columnas
columnas = dataset_gold_filtrado.columns.difference(['target'])
# Crear dict plano para el DataFrame de importancias
importancia_flat = {
    f"mean_{col}": importancia_media[idx]
    for idx, col in enumerate(columnas)
}
importancia_flat.update({
    f"std_{col}": importancia_std[idx]
    for idx, col in enumerate(columnas)
})
importancia_flat.update(params)  # Podés incluir los hiperparámetros para referencia

importancias_resultados.append(importancia_flat)

Evaluando parámetros: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}
Resultados para {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}:


In [19]:
resultados = pd.DataFrame(resultados)
resultados

,params,bootstrap,criterion,n_estimators,train_accuracy_mean,test_accuracy_mean,train_accuracy_std,test_accuracy_std,train_balanced_accuracy_mean,test_balanced_accuracy_mean,train_balanced_accuracy_std,test_balanced_accuracy_std,train_auc_mean,test_auc_mean,train_auc_std,test_auc_std
0,"{'bootstrap': True, 'criterion': 'entropy', 'm...",True,entropy,20,0.853764,0.851638,0.005071,0.011102,0.84713,0.845731,0.005315,0.012555,0.922613,0.919125,0.00014,0.001946


In [20]:
resultados.to_csv('../../assets/resultados_modelos/experimento_seleccion_variables/resultados_random_forest_20_arboles.csv', index=False)

Vemos para Gini

In [22]:
columnas_importantes_20_arboles_gini = ['tp_aprobados_3.0', 'tp_aprobados_2.0', 'finales_inscriptos_3.0', 'finales_inscriptos_2.0',
                                    'tp_aprobado_materia_6',
                                    'inscripciones_3.0', 'fecha_TP_materia_7', 'finales_inscriptos_1.0', 'inscripciones_2.0',
                                    'tp_aprobado_materia_3', 'nota_final_materia_5', 'tp_aprobado_materia_4', 'nota_final_materia_7',
                                    'fecha_final_materia_7', 'tp_aprobado_materia_8', 'tp_aprobado_materia_7',
                                    'nota_final_materia_6', 'nota_final_materia_3', 'nota_final_materia_4', 'fecha_final_materia_2',
                                     'target']

dataset_gold_filtrado_gini = dataset_gold[columnas_importantes_20_arboles_gini]
dataset_gold_filtrado_gini.isna().sum()

tp_aprobados_3.0             0
tp_aprobados_2.0             0
finales_inscriptos_3.0       0
finales_inscriptos_2.0       0
tp_aprobado_materia_6      879
inscripciones_3.0            0
fecha_TP_materia_7        1215
finales_inscriptos_1.0       0
inscripciones_2.0            0
tp_aprobado_materia_3      331
nota_final_materia_5      1596
tp_aprobado_materia_4      472
nota_final_materia_7      1786
fecha_final_materia_7     1754
tp_aprobado_materia_8     1532
tp_aprobado_materia_7     1215
nota_final_materia_6      1644
nota_final_materia_3      1440
nota_final_materia_4      1556
fecha_final_materia_2     1433
target                       0
dtype: int64

In [23]:
resultados = []
importancias_resultados = []
params = {'bootstrap': True,
  'criterion': 'entropy',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 20}


accuracy_scores_train = []
accuracy_scores_test = []
balanced_accuracy_scores_train = []
balanced_accuracy_scores_test = []
auc_scores_train = []
auc_scores_test = []
importancias_atributos = []
print(f"Evaluando parámetros: {params}")
# Repetir el proceso 5 veces para obtener diferentes splits de entrenamiento y test
for i in range(5):
    # 1. Separar un 20% random como test, el random state cambia en cada iteracion, entonces da un 20% distinto en cada una
    X_train_base, X_test, y_train_base, y_test = train_test_split(dataset_gold_filtrado_gini[dataset_gold_filtrado_gini.columns.difference(['target'])],
                                                                    dataset_gold_filtrado_gini['target'], test_size=0.2, random_state=i,
                                                                    stratify=dataset_gold_filtrado_gini['target'])
    
    modelo = RandomForestClassifier(random_state=42, **params)
    modelo.fit(X_train_base, y_train_base)

    # Guardar importancia de atributos
    importancias = modelo.feature_importances_
    importancias_atributos.append(importancias)
    
    # Predicciones de clase
    y_train_pred = modelo.predict(X_train_base)
    y_validation_pred = modelo.predict(X_test)

    # Predicciones de probabilidad para AUC ROC (para la clase positiva)
    y_train_proba = modelo.predict_proba(X_train_base)[:, 1]
    y_validation_proba = modelo.predict_proba(X_test)[:, 1]

    # Accuracy
    acc_train = accuracy_score(y_train_base, y_train_pred)
    acc_test = accuracy_score(y_test, y_validation_pred)
    accuracy_scores_train.append(acc_train)
    accuracy_scores_test.append(acc_test)

    # Balanced Accuracy
    bal_acc_train = balanced_accuracy_score(y_train_base, y_train_pred)
    bal_acc_test = balanced_accuracy_score(y_test, y_validation_pred)
    balanced_accuracy_scores_train.append(bal_acc_train)
    balanced_accuracy_scores_test.append(bal_acc_test)

    # AUC ROC
    auc_train = roc_auc_score(y_train_base, y_train_proba)
    auc_test = roc_auc_score(y_test, y_validation_proba)
    auc_scores_train.append(auc_train)
    auc_scores_test.append(auc_test)

# Calcular promedios y desviaciones estándar
accuracy_train_mean = np.mean(accuracy_scores_train)
accuracy_test_mean = np.mean(accuracy_scores_test)
accuracy_train_std = np.std(accuracy_scores_train)
accuracy_test_std = np.std(accuracy_scores_test)
bal_accuracy_train_mean = np.mean(balanced_accuracy_scores_train)
bal_accuracy_test_mean = np.mean(balanced_accuracy_scores_test)
bal_accuracy_train_std = np.std(balanced_accuracy_scores_train)
bal_accuracy_test_std = np.std(balanced_accuracy_scores_test)
auc_train_mean = np.mean(auc_scores_train)
auc_test_mean = np.mean(auc_scores_test)
auc_train_std = np.std(auc_scores_train)
auc_test_std = np.std(auc_scores_test)
print(f"Resultados para {params}:")

resultados.append({
    'params': params,
    'bootstrap': params['bootstrap'],
    'criterion': params['criterion'],
    'n_estimators': params['n_estimators'],
    'train_accuracy_mean': accuracy_train_mean,
    'test_accuracy_mean': accuracy_test_mean,
    'train_accuracy_std': accuracy_train_std,
    'test_accuracy_std': accuracy_test_std,
    'train_balanced_accuracy_mean': bal_accuracy_train_mean,
    'test_balanced_accuracy_mean': bal_accuracy_test_mean,
    'train_balanced_accuracy_std': bal_accuracy_train_std,
    'test_balanced_accuracy_std': bal_accuracy_test_std,
    'train_auc_mean': auc_train_mean,
    'test_auc_mean': auc_test_mean,
    'train_auc_std': auc_train_std,
    'test_auc_std': auc_test_std
})

# Calcular importancia promedio y std
importancias_array = np.array(importancias_atributos)
importancia_media = np.mean(importancias_array, axis=0)
importancia_std = np.std(importancias_array, axis=0)

# Asignar nombres de columnas
columnas = dataset_gold_filtrado_gini.columns.difference(['target'])
# Crear dict plano para el DataFrame de importancias
importancia_flat = {
    f"mean_{col}": importancia_media[idx]
    for idx, col in enumerate(columnas)
}
importancia_flat.update({
    f"std_{col}": importancia_std[idx]
    for idx, col in enumerate(columnas)
})
importancia_flat.update(params)  # Podés incluir los hiperparámetros para referencia

importancias_resultados.append(importancia_flat)

Evaluando parámetros: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}
Resultados para {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}:


In [24]:
resultados = pd.DataFrame(resultados)
resultados

,params,bootstrap,criterion,n_estimators,train_accuracy_mean,test_accuracy_mean,train_accuracy_std,test_accuracy_std,train_balanced_accuracy_mean,test_balanced_accuracy_mean,train_balanced_accuracy_std,test_balanced_accuracy_std,train_auc_mean,test_auc_mean,train_auc_std,test_auc_std
0,"{'bootstrap': True, 'criterion': 'entropy', 'm...",True,entropy,20,0.857432,0.852408,0.002214,0.010166,0.850866,0.847212,0.002423,0.010893,0.922699,0.920628,0.00116,0.002235


In [25]:
resultados.to_csv('../../assets/resultados_modelos/experimento_seleccion_variables/resultados_random_forest_20_arboles_gini.csv', index=False)